# LexData — Modelo Predictivo · Nicho Familiar · Demo

**Propuesta analítica implementada:**
1. Generación de datos sintéticos representativos (reemplazar con datos reales del CSJ)
2. Análisis exploratorio de variables críticas
3. Modelo de Regresión (XGBoost / Random Forest) para predicción de duración
4. Modelo de Survival Analysis (Cox PH) para probabilidad de resolución en el tiempo
5. Identificación de variables críticas (SHAP + importancia de características)
6. Evaluación del modelo (MAPE, MAE, curvas de supervivencia)
7. Exportación del modelo para uso en el dashboard Streamlit

**Inputs esperados:**
- `data_judicial/lexdata_co_ocurrencia_IVF_v6.csv` — generado por el notebook de scraping
- Datos de expedientes judiciales del CSJ (duración real de procesos) — ver nota de reemplazo

**Outputs:**
- `models/modelo_regresion.pkl` — modelo entrenado
- `models/modelo_cox.pkl` — modelo de supervivencia
- `models/feature_importance.csv` — variables críticas ordenadas por impacto
- `data_judicial/lexdata_expedientes_sinteticos.csv` — datos para la demo

## Sección 1 — Instalación y dependencias

In [ ]:
# Instalar dependencias si no están disponibles
# Descomenta y ejecuta solo si es necesario

# !pip install xgboost lifelines shap scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
import joblib

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

try:
    import xgboost as xgb
    XGB_DISPONIBLE = True
except ImportError:
    XGB_DISPONIBLE = False
    print("⚠ XGBoost no disponible — se usará GradientBoosting de sklearn")

try:
    from lifelines import CoxPHFitter, KaplanMeierFitter
    from lifelines.statistics import logrank_test
    LIFELINES_DISPONIBLE = True
except ImportError:
    LIFELINES_DISPONIBLE = False
    print("⚠ lifelines no disponible — instalar con: pip install lifelines")

try:
    import shap
    SHAP_DISPONIBLE = True
except ImportError:
    SHAP_DISPONIBLE = False
    print("⚠ SHAP no disponible — se usará importancia por permutación")

warnings.filterwarnings("ignore")
np.random.seed(42)

OUTPUT_DIR = "data_judicial"
MODEL_DIR = "models"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

print("✅ Librerías cargadas")
print(f"   XGBoost: {'✅' if XGB_DISPONIBLE else '⚠ no disponible'}")
print(f"   lifelines: {'✅' if LIFELINES_DISPONIBLE else '⚠ no disponible'}")
print(f"   SHAP: {'✅' if SHAP_DISPONIBLE else '⚠ no disponible'}")

## Sección 2 — Datos

### 2.1 Carga de IVF desde el pipeline de scraping

In [ ]:
# Intentar cargar datos reales del pipeline de scraping
ruta_ivf = f"{OUTPUT_DIR}/lexdata_co_ocurrencia_IVF_v6.csv"

if os.path.exists(ruta_ivf):
    df_ivf = pd.read_csv(ruta_ivf)
    print(f"✅ IVF cargado desde {ruta_ivf}: {df_ivf.shape}")
    print(df_ivf.head(3))
else:
    print(f"⚠ {ruta_ivf} no encontrado — ejecutar primero el notebook de scraping")
    print("  Continuando con datos sintéticos representativos para la demo...")
    df_ivf = None

### 2.2 Generación de expedientes sintéticos representativos

> **Nota de reemplazo:** Esta celda genera datos sintéticos basados en los patrones conocidos del sistema judicial colombiano (duración promedio por tipo de proceso, variabilidad por despacho, etc.). Para el modelo en producción, **reemplazar `df_expedientes` con los datos reales del CSJ** (dataset `x5yx-c7vy` de datos.gov.co o exportación directa de SICOF).

In [ ]:
# ── Parámetros del ciclo familiar en Valle del Cauca ─────────────────────────
# Basados en: Consejo Superior de la Judicatura — Informe Estadístico 2023
# y literatura académica de derecho de familia colombiano

MUNICIPIOS = [
    "CALI", "BUENAVENTURA", "PALMIRA", "TULUA", "JAMUNDI",
    "YUMBO", "GUADALAJARA DE BUGA", "CANDELARIA", "CARTAGO",
    "FLORIDA", "EL CERRITO", "PRADERA", "SEVILLA", "ZARZAL",
]

TIPOS_PROCESO = ["ALIMENTOS", "VIF", "HURTO_PATRIMONIAL", "SUSTANCIAS"]

DESPACHOS = [
    "Juzgado_1_Familia", "Juzgado_2_Familia", "Juzgado_3_Familia",
    "Comisaria_1_Familia", "Comisaria_2_Familia",
    "Juzgado_Penal_Municipal_1", "Juzgado_Penal_Municipal_2",
]

# Duración base en días por tipo de proceso (media, desviación)
# Fuente: estimaciones basadas en informes CSJ y DANE
DURACION_BASE = {
    "ALIMENTOS":        {"media": 240, "std": 90},
    "VIF":              {"media": 180, "std": 70},
    "HURTO_PATRIMONIAL":{"media": 280, "std": 110},
    "SUSTANCIAS":       {"media": 200, "std": 80},
}

# IVF score por municipio (basado en análisis del notebook de scraping)
IVF_MUNICIPIOS = {
    "CALI": 82, "BUENAVENTURA": 89, "PALMIRA": 74, "TULUA": 71,
    "JAMUNDI": 67, "YUMBO": 62, "GUADALAJARA DE BUGA": 53,
    "CANDELARIA": 48, "CARTAGO": 58, "FLORIDA": 44,
    "EL CERRITO": 41, "PRADERA": 38, "SEVILLA": 35, "ZARZAL": 33,
}

# Carga de despacho (mayor carga = mayor duración)
CARGA_DESPACHO = {
    "Juzgado_1_Familia": 1.35,
    "Juzgado_2_Familia": 1.20,
    "Juzgado_3_Familia": 0.95,
    "Comisaria_1_Familia": 1.10,
    "Comisaria_2_Familia": 0.90,
    "Juzgado_Penal_Municipal_1": 1.25,
    "Juzgado_Penal_Municipal_2": 1.05,
}

N_EXPEDIENTES = 5000  # Para demo; escalar a 500k con datos reales

registros = []
for i in range(N_EXPEDIENTES):
    municipio = np.random.choice(MUNICIPIOS)
    tipo = np.random.choice(TIPOS_PROCESO, p=[0.35, 0.35, 0.20, 0.10])
    despacho = np.random.choice(DESPACHOS)
    anio_rad = np.random.choice([2020, 2021, 2022, 2023, 2024], p=[0.10, 0.15, 0.20, 0.25, 0.30])

    ivf = IVF_MUNICIPIOS.get(municipio, 50)
    carga = CARGA_DESPACHO.get(despacho, 1.0)
    params = DURACION_BASE[tipo]

    # Duración modelada: base × carga_despacho + efecto_IVF + ruido
    duracion = (
        params["media"] * carga
        + ivf * 0.8                              # mayor vulnerabilidad → más demora
        + (2024 - anio_rad) * (-5)               # procesos más recientes son más ágiles
        + np.random.normal(0, params["std"])
    )
    duracion = max(30, int(duracion))  # mínimo 30 días

    # Evento: 1 = proceso terminado, 0 = censurado (aún activo)
    terminado = 1 if (duracion <= 365 * 2 and np.random.random() > 0.15) else 0
    if not terminado:
        duracion = int(duracion * np.random.uniform(0.4, 0.9))  # censurado antes del cierre

    # Número de audiencias (correlaciona con duración)
    n_audiencias = max(1, int(duracion / 45 + np.random.normal(0, 1)))

    # Apelación (aumenta duración)
    apelacion = 1 if (tipo in ["ALIMENTOS", "HURTO_PATRIMONIAL"] and np.random.random() > 0.65) else 0
    if apelacion:
        duracion = int(duracion * 1.4)

    registros.append({
        "expediente_id": f"EXP-{i+1:05d}",
        "municipio": municipio,
        "tipo_proceso": tipo,
        "despacho": despacho,
        "anio_radicacion": anio_rad,
        "ivf_score": ivf,
        "carga_despacho": carga,
        "n_audiencias": n_audiencias,
        "apelacion": apelacion,
        "duracion_dias": duracion,
        "terminado": terminado,
    })

df_exp = pd.DataFrame(registros)

# Guardar
ruta_exp = f"{OUTPUT_DIR}/lexdata_expedientes_sinteticos.csv"
df_exp.to_csv(ruta_exp, index=False)

print(f"✅ Expedientes generados: {len(df_exp):,}")
print(f"   Duración media: {df_exp['duracion_dias'].mean():.0f} días")
print(f"   Terminados: {df_exp['terminado'].mean()*100:.1f}%")
print(f"💾 Guardado: {ruta_exp}")
print()
df_exp.groupby("tipo_proceso")["duracion_dias"].agg(["mean", "median", "std"]).round(1)

## Sección 3 — Análisis Exploratorio (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("EDA — Duración de procesos familiares · Valle del Cauca", fontsize=13, fontweight="bold")

# 1. Distribución de duración por tipo
ax = axes[0, 0]
for tipo in TIPOS_PROCESO:
    subset = df_exp[df_exp["tipo_proceso"] == tipo]["duracion_dias"]
    ax.hist(subset, bins=40, alpha=0.6, label=tipo, edgecolor="none")
ax.set_xlabel("Duración (días)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de duración por tipo de proceso")
ax.legend(fontsize=9)

# 2. Duración media por municipio (top 10)
ax = axes[0, 1]
mun_dur = df_exp.groupby("municipio")["duracion_dias"].mean().sort_values(ascending=True).tail(10)
colors = ["#A32D2D" if v > mun_dur.quantile(0.75) else "#BA7517" if v > mun_dur.quantile(0.5) else "#3B6D11"
          for v in mun_dur.values]
ax.barh(mun_dur.index, mun_dur.values, color=colors, edgecolor="none")
ax.set_xlabel("Duración media (días)")
ax.set_title("Top 10 municipios por duración media")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x)}"))

# 3. IVF vs Duración (scatter)
ax = axes[1, 0]
sample = df_exp.sample(min(800, len(df_exp)), random_state=42)
scatter_colors = {"ALIMENTOS": "#E24B4A", "VIF": "#BA7517", "HURTO_PATRIMONIAL": "#378ADD", "SUSTANCIAS": "#1D9E75"}
for tipo, color in scatter_colors.items():
    mask = sample["tipo_proceso"] == tipo
    ax.scatter(sample[mask]["ivf_score"], sample[mask]["duracion_dias"],
               c=color, alpha=0.4, s=15, label=tipo)
# Línea de tendencia
z = np.polyfit(sample["ivf_score"], sample["duracion_dias"], 1)
p = np.poly1d(z)
x_line = np.linspace(sample["ivf_score"].min(), sample["ivf_score"].max(), 100)
ax.plot(x_line, p(x_line), "k--", linewidth=1.5, label="Tendencia")
ax.set_xlabel("IVF Score")
ax.set_ylabel("Duración (días)")
ax.set_title("Correlación IVF → Duración del proceso")
ax.legend(fontsize=8)

# 4. Duración por despacho (boxplot)
ax = axes[1, 1]
despacho_labels = [d.replace("_", " ").replace("Juzgado", "Jdo.").replace("Comisaria", "Com.") for d in DESPACHOS]
data_boxes = [df_exp[df_exp["despacho"] == d]["duracion_dias"].values for d in DESPACHOS]
bp = ax.boxplot(data_boxes, vert=True, patch_artist=True, notch=False,
                medianprops={"color": "black", "linewidth": 1.5})
for patch in bp["boxes"]:
    patch.set_facecolor("#B5D4F4")
    patch.set_alpha(0.7)
ax.set_xticklabels(despacho_labels, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Duración (días)")
ax.set_title("Variabilidad de duración por despacho")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/eda_duracion_familiar.png", dpi=110, bbox_inches="tight")
plt.show()
print("✅ EDA completado")

## Sección 4 — Modelo de Regresión (Predicción de duración)

In [ ]:
# ── Preparar features ─────────────────────────────────────────────────────────

FEATURES = ["ivf_score", "carga_despacho", "n_audiencias", "apelacion",
            "anio_radicacion", "tipo_proceso_enc", "municipio_enc", "despacho_enc"]

df_model = df_exp.copy()

# Encoding de variables categóricas
le_tipo = LabelEncoder()
le_mun = LabelEncoder()
le_desp = LabelEncoder()

df_model["tipo_proceso_enc"] = le_tipo.fit_transform(df_model["tipo_proceso"])
df_model["municipio_enc"] = le_mun.fit_transform(df_model["municipio"])
df_model["despacho_enc"] = le_desp.fit_transform(df_model["despacho"])

X = df_model[FEATURES]
y = df_model["duracion_dias"]

# Split temporal: train 2020-2023, test 2024 (validación realista)
X_train = X[df_model["anio_radicacion"] < 2024]
y_train = y[df_model["anio_radicacion"] < 2024]
X_test  = X[df_model["anio_radicacion"] == 2024]
y_test  = y[df_model["anio_radicacion"] == 2024]

print(f"Split temporal:")
print(f"  Train (2020-2023): {len(X_train):,} expedientes")
print(f"  Test  (2024):      {len(X_test):,} expedientes")

In [ ]:
# ── Entrenar modelos ─────────────────────────────────────────────────────────

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — métrica objetivo de LexData (≤ 15%)."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


modelos = {
    "Regresión Lineal": Ridge(alpha=1.0),
    "Random Forest":    RandomForestRegressor(n_estimators=200, max_depth=8, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42),
}
if XGB_DISPONIBLE:
    modelos["XGBoost"] = xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.04, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    )

resultados = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_pred = np.maximum(y_pred, 0)  # duración no puede ser negativa

    mae_val  = mean_absolute_error(y_test, y_pred)
    mape_val = mape(y_test, y_pred)
    r2_val   = r2_score(y_test, y_pred)
    rmse_val = np.sqrt(mean_squared_error(y_test, y_pred))

    resultados[nombre] = {"MAE": mae_val, "MAPE": mape_val, "R2": r2_val, "RMSE": rmse_val, "modelo": modelo}
    alerta = "✅" if mape_val <= 15 else "⚠"
    print(f"{alerta} {nombre:<22} MAPE={mape_val:.1f}%  MAE={mae_val:.0f}d  R²={r2_val:.3f}")

# Seleccionar mejor modelo por MAPE
mejor_nombre = min(resultados, key=lambda k: resultados[k]["MAPE"])
mejor_modelo = resultados[mejor_nombre]["modelo"]
mejor_mape   = resultados[mejor_nombre]["MAPE"]

print(f"\n🏆 Mejor modelo: {mejor_nombre} (MAPE={mejor_mape:.1f}%)")
print(f"   {'✅ Cumple' if mejor_mape <= 15 else '⚠ No cumple'} objetivo MAPE ≤ 15%")

In [ ]:
# ── Guardar modelo ────────────────────────────────────────────────────────────
modelo_path = f"{MODEL_DIR}/modelo_regresion.pkl"
joblib.dump({
    "modelo": mejor_modelo,
    "nombre": mejor_nombre,
    "features": FEATURES,
    "le_tipo": le_tipo,
    "le_mun": le_mun,
    "le_desp": le_desp,
    "mape": mejor_mape,
}, modelo_path)
print(f"💾 Modelo guardado: {modelo_path}")

# ── Visualizar predicciones vs reales ─────────────────────────────────────────
y_pred_best = mejor_modelo.predict(X_test)
y_pred_best = np.maximum(y_pred_best, 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"Evaluación del modelo — {mejor_nombre}", fontsize=12, fontweight="bold")

# Predicho vs real
ax = axes[0]
max_val = max(y_test.max(), y_pred_best.max())
ax.scatter(y_test, y_pred_best, alpha=0.3, s=10, color="#378ADD")
ax.plot([0, max_val], [0, max_val], "r--", linewidth=1.5, label="Predicción perfecta")
ax.set_xlabel("Duración real (días)")
ax.set_ylabel("Duración predicha (días)")
ax.set_title("Predicho vs Real")
ax.legend()
ax.text(0.05, 0.93, f"MAPE = {mejor_mape:.1f}%\nR² = {resultados[mejor_nombre]['R2']:.3f}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

# Distribución de errores
ax = axes[1]
errores = y_pred_best - np.array(y_test)
ax.hist(errores, bins=50, color="#5DCAA5", edgecolor="none", alpha=0.8)
ax.axvline(0, color="red", linestyle="--", linewidth=1.5)
ax.set_xlabel("Error (días predichos - días reales)")
ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de errores del modelo")
ax.text(0.65, 0.90, f"Media: {errores.mean():.1f}d\nDesv: {errores.std():.1f}d",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.5))

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/evaluacion_modelo_regresion.png", dpi=110, bbox_inches="tight")
plt.show()

## Sección 5 — Identificación de Variables Críticas

In [ ]:
# ── Importancia de variables ──────────────────────────────────────────────────

FEATURE_LABELS = {
    "ivf_score":        "IVF Score (vulnerabilidad)",
    "carga_despacho":   "Carga del despacho",
    "n_audiencias":     "Número de audiencias",
    "apelacion":        "Con apelación (sí/no)",
    "anio_radicacion":  "Año de radicación",
    "tipo_proceso_enc": "Tipo de proceso",
    "municipio_enc":    "Municipio",
    "despacho_enc":     "Despacho asignado",
}

# Método 1: importancia nativa del modelo (para árbol/XGBoost)
if hasattr(mejor_modelo, "feature_importances_"):
    importancias = mejor_modelo.feature_importances_
    metodo = "Importancia nativa (ganancia de información)"
else:
    # Método 2: importancia por permutación (agnóstico al modelo)
    perm = permutation_importance(mejor_modelo, X_test, y_test, n_repeats=15, random_state=42)
    importancias = perm.importances_mean
    importancias = np.maximum(importancias, 0)  # ignorar negativas
    metodo = "Importancia por permutación"

# Normalizar a porcentaje
if importancias.sum() > 0:
    importancias_pct = importancias / importancias.sum() * 100
else:
    importancias_pct = np.ones(len(FEATURES)) / len(FEATURES) * 100

df_importancia = pd.DataFrame({
    "feature": FEATURES,
    "label": [FEATURE_LABELS.get(f, f) for f in FEATURES],
    "importancia_pct": importancias_pct.round(2),
}).sort_values("importancia_pct", ascending=False).reset_index(drop=True)

df_importancia.to_csv(f"{MODEL_DIR}/feature_importance.csv", index=False)

print(f"✅ Variables críticas ({metodo}):")
print()
for _, row in df_importancia.iterrows():
    bar = "█" * int(row["importancia_pct"] / 2)
    print(f"  {row['label']:<35} {bar} {row['importancia_pct']:.1f}%")

# Visualización
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#A32D2D" if i < 2 else "#BA7517" if i < 4 else "#3B6D11" for i in range(len(df_importancia))]
bars = ax.barh(df_importancia["label"][::-1], df_importancia["importancia_pct"][::-1],
               color=colors[::-1], edgecolor="none")
ax.set_xlabel("Importancia relativa (%)")
ax.set_title(f"Variables críticas que explican la duración del proceso\n({metodo})", fontsize=11)
for bar, val in zip(bars, df_importancia["importancia_pct"][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/feature_importance.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# ── SHAP (si está disponible) ─────────────────────────────────────────────────

if SHAP_DISPONIBLE and (XGB_DISPONIBLE and isinstance(mejor_modelo, xgb.XGBRegressor)):
    print("Calculando valores SHAP...")
    explainer = shap.Explainer(mejor_modelo, X_train)
    shap_values = explainer(X_test.sample(min(300, len(X_test)), random_state=42))

    shap.summary_plot(
        shap_values, feature_names=[FEATURE_LABELS.get(f, f) for f in FEATURES],
        show=False, plot_size=(10, 5)
    )
    plt.title("SHAP — Impacto de cada variable en la predicción")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/shap_summary.png", dpi=110, bbox_inches="tight")
    plt.show()
    print("✅ SHAP calculado")
else:
    print("ℹ SHAP omitido (no disponible o modelo no compatible — se usó importancia por permutación)")

## Sección 6 — Survival Analysis (Cox Proportional Hazards)

In [ ]:
if not LIFELINES_DISPONIBLE:
    print("⚠ lifelines no disponible. Instalar con: pip install lifelines")
else:
    # ── Kaplan-Meier por tipo de proceso ──────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Survival Analysis — Nicho Familiar · Valle del Cauca", fontsize=12, fontweight="bold")

    # Kaplan-Meier
    ax = axes[0]
    kmf = KaplanMeierFitter()
    colores_km = {"ALIMENTOS": "#E24B4A", "VIF": "#BA7517", "HURTO_PATRIMONIAL": "#378ADD", "SUSTANCIAS": "#1D9E75"}

    for tipo in TIPOS_PROCESO:
        subset = df_exp[df_exp["tipo_proceso"] == tipo]
        kmf.fit(subset["duracion_dias"], event_observed=subset["terminado"], label=tipo)
        kmf.plot_survival_function(ax=ax, ci_show=False, color=colores_km[tipo], linewidth=2)

    ax.set_xlabel("Días desde radicación")
    ax.set_ylabel("P(proceso aún activo)")
    ax.set_title("Curvas Kaplan-Meier por tipo de proceso")
    ax.set_xlim(0, 900)
    ax.axhline(0.5, color="gray", linestyle=":", linewidth=1, label="Mediana (50%)")
    ax.legend(fontsize=9)

    # Kaplan-Meier por nivel de IVF
    ax = axes[1]
    df_exp["ivf_grupo"] = pd.cut(
        df_exp["ivf_score"], bins=[0, 50, 70, 100],
        labels=["IVF Bajo (<50)", "IVF Medio (50-70)", "IVF Alto (>70)"]
    )
    colores_ivf = {"IVF Bajo (<50)": "#3B6D11", "IVF Medio (50-70)": "#BA7517", "IVF Alto (>70)": "#A32D2D"}

    for grupo in ["IVF Bajo (<50)", "IVF Medio (50-70)", "IVF Alto (>70)"]:
        subset = df_exp[df_exp["ivf_grupo"] == grupo]
        if len(subset) > 0:
            kmf.fit(subset["duracion_dias"], event_observed=subset["terminado"], label=grupo)
            kmf.plot_survival_function(ax=ax, ci_show=False, color=colores_ivf[grupo], linewidth=2)

    ax.set_xlabel("Días desde radicación")
    ax.set_ylabel("P(proceso aún activo)")
    ax.set_title("Efecto del IVF en la duración del proceso")
    ax.set_xlim(0, 900)
    ax.axhline(0.5, color="gray", linestyle=":", linewidth=1)
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/survival_kaplan_meier.png", dpi=110, bbox_inches="tight")
    plt.show()
    print("✅ Curvas Kaplan-Meier generadas")

In [ ]:
if not LIFELINES_DISPONIBLE:
    print("⚠ lifelines no disponible — omitiendo Cox PH")
else:
    # ── Cox Proportional Hazards ──────────────────────────────────────────────
    # El modelo Cox estima el riesgo de resolución en función de covariables.
    # HR > 1: la variable acelera la resolución
    # HR < 1: la variable retrasa la resolución

    df_cox = df_model[["duracion_dias", "terminado", "ivf_score",
                        "carga_despacho", "n_audiencias", "apelacion",
                        "anio_radicacion", "tipo_proceso_enc"]].copy()

    # Normalizar covariables numéricas
    scaler = StandardScaler()
    cols_escalar = ["ivf_score", "carga_despacho", "n_audiencias", "anio_radicacion"]
    df_cox[cols_escalar] = scaler.fit_transform(df_cox[cols_escalar])

    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(df_cox, duration_col="duracion_dias", event_col="terminado")

    print("\n── Resultados Cox PH ──")
    cph.print_summary(decimals=3)

    # Guardar modelo
    cox_path = f"{MODEL_DIR}/modelo_cox.pkl"
    joblib.dump({"modelo": cph, "scaler": scaler, "cols_escalar": cols_escalar}, cox_path)
    print(f"\n💾 Modelo Cox guardado: {cox_path}")

    # Visualizar hazard ratios
    fig, ax = plt.subplots(figsize=(8, 5))
    cph.plot(ax=ax)
    ax.set_title("Cox PH — Hazard Ratios (HR > 1 = resuelve más rápido)")
    ax.axvline(0, color="red", linestyle="--", linewidth=1)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/cox_hazard_ratios.png", dpi=110, bbox_inches="tight")
    plt.show()
    print("✅ Modelo Cox PH entrenado")

## Sección 7 — Sistema de Alertas Tempranas

In [ ]:
# ── Detectar casos en riesgo de retraso ───────────────────────────────────────
# Un caso está en riesgo si su duración estimada supera el percentil 75
# histórico del mismo tipo de proceso en el mismo despacho.

# Percentiles históricos por tipo de proceso y despacho
percentiles = (
    df_exp
    .groupby(["tipo_proceso", "despacho"])["duracion_dias"]
    .quantile(0.75)
    .reset_index()
    .rename(columns={"duracion_dias": "p75_duracion"})
)

# Predecir duración para todos los expedientes
df_alerta = df_model.copy()
df_alerta["duracion_estimada"] = np.maximum(mejor_modelo.predict(X), 0).round().astype(int)

# Unir percentiles
df_alerta = df_alerta.merge(percentiles, on=["tipo_proceso", "despacho"], how="left")

# Definir riesgo
df_alerta["riesgo"] = "Bajo"
df_alerta.loc[df_alerta["duracion_estimada"] > df_alerta["p75_duracion"], "riesgo"] = "Medio"
df_alerta.loc[df_alerta["duracion_estimada"] > df_alerta["p75_duracion"] * 1.5, "riesgo"] = "Alto"

# Exportar alertas
df_alertas_activas = df_alerta[df_alerta["riesgo"] != "Bajo"].sort_values(
    ["riesgo", "duracion_estimada"], ascending=[True, False]
)

ruta_alertas = f"{OUTPUT_DIR}/lexdata_alertas_tempranas.csv"
df_alertas_activas.to_csv(ruta_alertas, index=False)

print("── RESUMEN DE ALERTAS TEMPRANAS ──")
print(df_alerta["riesgo"].value_counts().to_string())
print(f"\n💾 Alertas guardadas: {ruta_alertas}")
print()

# Muestra de alertas de riesgo alto
cols_mostrar = ["expediente_id", "municipio", "tipo_proceso", "despacho",
                "ivf_score", "duracion_estimada", "p75_duracion", "riesgo"]
print("Top 10 casos en riesgo ALTO:")
print(df_alertas_activas[df_alertas_activas["riesgo"] == "Alto"][cols_mostrar].head(10).to_string(index=False))

## Sección 8 — Resumen de Outputs

In [ ]:
print("=" * 60)
print("OUTPUTS GENERADOS — LexData Modelo Predictivo Demo")
print("=" * 60)

outputs = [
    (f"{OUTPUT_DIR}/lexdata_expedientes_sinteticos.csv", "Expedientes sintéticos (reemplazar con CSJ)"),
    (f"{OUTPUT_DIR}/eda_duracion_familiar.png",          "Gráficas EDA"),
    (f"{OUTPUT_DIR}/evaluacion_modelo_regresion.png",    "Evaluación modelo regresión"),
    (f"{OUTPUT_DIR}/feature_importance.png",             "Variables críticas"),
    (f"{OUTPUT_DIR}/survival_kaplan_meier.png",          "Curvas Kaplan-Meier"),
    (f"{OUTPUT_DIR}/cox_hazard_ratios.png",              "Cox PH — Hazard Ratios"),
    (f"{OUTPUT_DIR}/lexdata_alertas_tempranas.csv",      "Alertas tempranas activas"),
    (f"{MODEL_DIR}/modelo_regresion.pkl",                "Modelo de regresión serializado"),
    (f"{MODEL_DIR}/modelo_cox.pkl",                      "Modelo Cox PH serializado"),
    (f"{MODEL_DIR}/feature_importance.csv",              "Importancia de variables"),
]

for ruta, desc in outputs:
    existe = "✅" if os.path.exists(ruta) else "⚠ no generado"
    tam = f"({os.path.getsize(ruta)/1024:.0f} KB)" if os.path.exists(ruta) else ""
    print(f"  {existe} {desc:<45} {tam}")

print()
print("MÉTRICAS DEL MODELO:")
print(f"  Mejor modelo: {mejor_nombre}")
print(f"  MAPE:  {resultados[mejor_nombre]['MAPE']:.1f}%  (objetivo: ≤ 15%)")
print(f"  MAE:   {resultados[mejor_nombre]['MAE']:.0f} días")
print(f"  R²:    {resultados[mejor_nombre]['R2']:.3f}")
print(f"  RMSE:  {resultados[mejor_nombre]['RMSE']:.0f} días")
print()
print("SIGUIENTE PASO: ejecutar streamlit/app.py para el dashboard interactivo")
print("  streamlit run streamlit/app.py")
print("=" * 60)